# LLM-as-Jev: Qwen3-0.6B full fine-tune (Colab)

Run the cells top to bottom. Every stage checks the Hugging Face repos first and skips or resumes if its artifact exists.
The HF token is read from the Colab secret `my_hf_token` and is never printed.
Works on A100, L4 and T4 (batch plan and precision are chosen automatically).

## Setup

In [ ]:
import os, sys, time, json, subprocess
from pathlib import Path
if Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content")
    if not Path("llm-as-jev").exists():
        subprocess.run(["git", "clone", "https://github.com/hasnain1241/llm-as-jev"], check=True)
    os.chdir("llm-as-jev")
    subprocess.run(["git", "pull", "--ff-only"], check=False)
print("cwd:", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
sys.path.insert(0, os.getcwd())
from llm_as_jev.auth import login_hf
from llm_as_jev import hub, data, train, eval as jeval

login_hf()                      # token comes from the environment or Colab secret, never printed
hub.ensure_repos()
cfg = train.load_config("configs/qwen3_0p6b.yaml")
plan = train.gpu_plan()
os.makedirs("data", exist_ok=True); os.makedirs("outputs", exist_ok=True)
print({k: (str(v) if "dtype" in k else v) for k, v in plan.items()})

NB_START, CU_TOTAL = time.time(), 0.0
def report(stage, t0):
    """Print GPU, memory, elapsed time and a rough compute-unit estimate for a stage."""
    global CU_TOTAL
    dt = time.time() - t0
    cu = dt / 3600 * plan["cu_per_hour"]
    CU_TOTAL += cu
    print(f"[{stage}] GPU={plan['name']}  {train.mem_str()}  elapsed={dt/60:.1f} min  "
          f"~{cu:.2f} CU (running total ~{CU_TOTAL:.2f} CU, rough estimate)")

## Stage 1: smoke tests (trie, parallel scoring vs naive, loss, token never printed)

In [ ]:
t0 = time.time()
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-x"], capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-1500:])
assert r.returncode == 0, "Tests failed, fix before spending compute."
report("stage 1", t0)

## Stage 2: build and push Reason-9k, then the training-free baseline

In [ ]:
t0 = time.time()
tok = train.load_tokenizer(cfg["model"])

p = hub.pull_dataset_file("processed/eval_sets.json")
if p:
    eval_sets = json.load(open(p)); print("eval sets pulled from HF")
else:
    eval_sets = data.load_eval_sets(cfg["seed"])
    json.dump(eval_sets, open("data/eval_sets.json", "w"))
    hub.push_dataset_file("data/eval_sets.json", "processed/eval_sets.json")
print({k: len(v) for k, v in eval_sets.items()})

p = hub.pull_dataset_file("processed/reason9k.jsonl")
if p:
    items = data.read_jsonl(p); print("Reason-9k pulled from HF:", len(items))
else:
    items = data.build_reason9k(tok, eval_sets, cfg["seed"], cfg["per_set"], cfg["max_len"])
    data.write_jsonl(items, "data/reason9k.jsonl")
    hub.push_dataset_file("data/reason9k.jsonl", "processed/reason9k.jsonl")
    hub.push_json(data.SOURCES, "processed/dataset_sources.json")
    print("dataset sources used:"); print(json.dumps(data.SOURCES, indent=1))
print(len(items), "training items")

In [ ]:
JEVBENCH_PATH = None   # set to a jsonl path (state, options, label) to include JevBench
jev = data.load_jevbench(JEVBENCH_PATH) if JEVBENCH_PATH else None

base = hub.pull_json("results/baseline_training_free.json")
if base is None:
    model = train.load_model(cfg["model"], plan["eval_dtype"])
    base = jeval.run_eval(model, tok, eval_sets, cfg["text_mode_n"], cfg["seed"], jevbench=jev)
    hub.push_json(base, "results/baseline_training_free.json")
    del model; torch.cuda.empty_cache()
else:
    print("baseline pulled from HF")
print(json.dumps({k: base[k] for k in ("external_macro", "banking77", "text_mode_total")}, indent=1))
report("stage 2", t0)

## Stage 3: cache top-64 reference statistics with the frozen model

In [ ]:
t0 = time.time()
p = hub.pull_dataset_file("cache/ref_stats_top64.pt")
if p:
    refs = torch.load(p, weights_only=False); print("reference stats pulled from HF")
else:
    frozen = train.load_model(cfg["model"], torch.float32)
    refs = train.build_reference_cache(frozen, tok, items, cfg["seed"], plan)
    os.makedirs("data/cache", exist_ok=True)
    torch.save(refs, "data/cache/ref_stats_top64.pt")
    hub.push_dataset_file("data/cache/ref_stats_top64.pt", "cache/ref_stats_top64.pt")
    del frozen; torch.cuda.empty_cache()
assert len(refs) == len(items)
report("stage 3", t0)

## Stage 4: full fine-tune
Checkpoints go to Google Drive (exact resume) and to the HF model repo (`checkpoint-latest`).
After 10 steps the projected compute units are printed; if above the budget the run stops here.

In [ ]:
t0 = time.time()
try:
    model, total_steps = train.train(cfg, items, refs, tok, plan)
except train.BudgetExceeded as e:
    print("STOP:", e)
    print("Ask the owner before continuing (set approve_over_budget: true in the config).")
    raise
report("stage 4", t0)

## Stage 5: evaluate, compare, push final model, write results table

In [ ]:
t0 = time.time()
model.eval(); model.config.use_cache = True
model.gradient_checkpointing_disable()
model.to(plan["eval_dtype"])
ft = hub.pull_json("results/finetuned.json")
if ft is None:
    ft = jeval.run_eval(model, tok, eval_sets, cfg["text_mode_n"], cfg["seed"], jevbench=jev)
    hub.push_json(ft, "results/finetuned.json")
table = jeval.results_table(base, ft)
print(table)
hub.push_json({"table": table}, "results/results_table.json")

# final model + model card to the HF model repo
final_dir = "outputs/final"
os.makedirs(final_dir, exist_ok=True)
train._bf16_copy_save(model, final_dir, tok)
hub.push_final_model(final_dir, hub.build_model_card(table))

# results table into README.md (between markers)
readme = open("README.md").read()
a, b = "<!-- RESULTS_START -->", "<!-- RESULTS_END -->"
if a in readme and b in readme:
    head, rest = readme.split(a); _, tail = rest.split(b)
    open("README.md", "w").write(head + a + "\n" + table + "\n" + b + tail)
    print("README.md updated in the Colab clone; commit and push it from your machine.")
report("stage 5", t0)
print(f"Total wall time {(time.time()-NB_START)/60:.1f} min, ~{CU_TOTAL:.2f} CU (rough)")

## Try it yourself (demo)
Loads the base model and the fine-tuned model from the HF model repo and shows each model's probability distribution over your options.
Runs on its own after the setup cells, no training needed.

In [ ]:
from transformers import AutoModelForCausalLM
from llm_as_jev import demo
tok = train.load_tokenizer(cfg["model"])
dt = plan["eval_dtype"]
models = {
    "Base Qwen3-0.6B (training-free)": AutoModelForCausalLM.from_pretrained(cfg["model"], dtype=dt).cuda().eval(),
    "Fine-tuned LLM-as-Jev": AutoModelForCausalLM.from_pretrained(hub.MODEL_REPO, dtype=dt).cuda().eval(),
}
demo.show(models, tok, "I think my card was stolen and I see payments I did not make",
          ["card arrival", "lost or stolen card", "exchange rate", "top up failed"])
demo.show(models, tok, "A man puts a pot of water on a lit stove. What happens next?",
          ["The water freezes", "The water heats up and boils", "The pot turns into glass"])

In [ ]:
demo.ask(models, tok)   # type your own state and options; re-run this cell as often as you like

## JevBench public items (evaluation only)
Clones JevBench (tag v1.4.2.2, falls back to main) into the git-ignored `data/jevbench`, converts the 231 public items to our template
(canonical option order, label plus rubric description per option) and scores the base and fine-tuned models.
These items are never used for training or checkpoint selection. Results go to `results/jevbench.json` in the dataset repo.

In [ ]:
from transformers import AutoModelForCausalLM
from llm_as_jev import jevbench
tok = train.load_tokenizer(cfg["model"])
version = jevbench.fetch_jevbench("data/jevbench")
jb = jevbench.load_public("data/jevbench")
print(version, len(jb), {t: sum(i["meta"]["tier"] == t for i in jb) for t in jevbench.TIERS})

res = hub.pull_json("results/jevbench.json")
if res is None:
    train_items = data.read_jsonl(hub.pull_dataset_file("processed/reason9k.jsonl"))
    dt = plan["eval_dtype"]
    mods = {
        "base": AutoModelForCausalLM.from_pretrained(cfg["model"], dtype=dt).cuda().eval(),
        "fine_tuned": AutoModelForCausalLM.from_pretrained(hub.MODEL_REPO, dtype=dt).cuda().eval(),
    }
    res = jevbench.evaluate(mods, tok, jb, train_items, version)
    hub.push_json(res, "results/jevbench.json")
    del mods; torch.cuda.empty_cache()
print("train items sharing a 13-gram with JevBench:", res.get("train_items_overlapping_jevbench_13gram"))
md_table = jevbench.markdown(res)
print(md_table)
hub.push_json({"table": md_table}, "results/jevbench_table.json")

## Share image (one PNG with both result tables)
Reads the saved results from the HF dataset repo, so the numbers match the README, and renders them as `outputs/results_card.png`.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

base = hub.pull_json("results/baseline_training_free.json")
ft = hub.pull_json("results/finetuned.json")
jb = hub.pull_json("results/jevbench.json")["models"]
jb_base, jb_ft = jb["base"], jb["fine_tuned"]

LONG = "long_over_1500_tokens"
ext = [  # label, base, fine-tuned, higher_is_better, format
    ("Accuracy, macro of 6 sets", base["external_macro"]["acc"], ft["external_macro"]["acc"], True, "{:.3f}"),
    ("ECE, macro of 6 sets", base["external_macro"]["ece"], ft["external_macro"]["ece"], False, "{:.3f}"),
    ("Brier, macro of 6 sets", base["external_macro"]["brier"], ft["external_macro"]["brier"], False, "{:.3f}"),
    ("Banking77 accuracy (77 options)", base["banking77"]["acc"], ft["banking77"]["acc"], True, "{:.3f}"),
    ("Banking77 ECE", base["banking77"]["ece"], ft["banking77"]["ece"], False, "{:.3f}"),
]
jev = [
    ("Overall accuracy (231 items)", jb_base["overall"]["acc"], jb_ft["overall"]["acc"], True, "{:.3f}"),
    ("Easy (48)", jb_base["tiers"]["easy"]["acc"], jb_ft["tiers"]["easy"]["acc"], True, "{:.3f}"),
    ("Standard (72)", jb_base["tiers"]["standard"]["acc"], jb_ft["tiers"]["standard"]["acc"], True, "{:.3f}"),
    ("Hard (111)", jb_base["tiers"]["hard"]["acc"], jb_ft["tiers"]["hard"]["acc"], True, "{:.3f}"),
    ("Prompts over 1500 tokens (37)", jb_base[LONG]["acc"], jb_ft[LONG]["acc"], True, "{:.3f}"),
    ("ECE", jb_base["overall"]["ece"], jb_ft["overall"]["ece"], False, "{:.3f}"),
    ("Brier", jb_base["overall"]["brier"], jb_ft["overall"]["brier"], False, "{:.3f}"),
]

def draw(ax, title, rows, c0, c1):
    ax.axis("off")
    ax.set_title(title, loc="left", fontsize=15, fontweight="bold", pad=10)
    cells = [[r[0], r[4].format(r[1]), r[4].format(r[2])] for r in rows]
    t = ax.table(cellText=cells, colLabels=["Metric", c0, c1], cellLoc="center", colLoc="center",
                 colWidths=[0.52, 0.24, 0.24], loc="upper left")
    t.auto_set_font_size(False); t.set_fontsize(13); t.scale(1, 1.9)
    for (i, j), cell in t.get_celld().items():
        cell.set_edgecolor("#d0d5dd")
        if i == 0:
            cell.set_facecolor("#1f2937"); cell.set_text_props(color="white", fontweight="bold")
        elif j == 0:
            cell.set_text_props(ha="left"); cell._loc = "left"
        if i > 0 and j > 0:
            r = rows[i - 1]
            better_ft = (r[2] > r[1]) if r[3] else (r[2] < r[1])
            if (j == 2) == better_ft and r[1] != r[2]:
                cell.set_facecolor("#e7f5ec"); cell.set_text_props(fontweight="bold")

fig, axes = plt.subplots(2, 1, figsize=(10, 11.2), gridspec_kw={"height_ratios": [len(ext) + 1.4, len(jev) + 1.4]})
fig.suptitle("LLM-as-Jev: Qwen3-0.6B full fine-tune (Reason-9k)", fontsize=18, fontweight="bold", x=0.04, ha="left", y=0.985)
draw(axes[0], "External benchmarks (training-free vs fine-tuned)", ext, "Training-free", "Fine-tuned")
draw(axes[1], "JevBench public items (base vs fine-tuned)", jev, "Base", "Fine-tuned")
fig.text(0.04, 0.012, "Green = better. Lower is better for ECE and Brier. Single run, one seed. JevBench: 231 public items,
"
         "not the official score. Accuracy differences of a few items are within noise; the calibration gain is the consistent effect.
"
         "github.com/hasnain1241/llm-as-jev", fontsize=10, color="#475467", va="bottom")
fig.subplots_adjust(top=0.92, bottom=0.09, hspace=0.25, left=0.04, right=0.96)
os.makedirs("outputs", exist_ok=True)
fig.savefig("outputs/results_card.png", dpi=200, facecolor="white")
plt.show()
try:
    from google.colab import files
    files.download("outputs/results_card.png")
except Exception:
    print("Saved to outputs/results_card.png")

## Re-run the test suite

In [ ]:
!python -m pytest tests -q